<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# PySpark Performance and Optimisation

*Session 5 · Notebook 02.04 · Lecture · Student version*

## Overview

So far we have focused on getting the **right answer**. On real volumes, getting it in reasonable **time and cost** matters just as much. This notebook opens the bonnet: how Spark turns your code into an execution plan, where the expensive steps are (the **shuffles**), and the four habits that fix most slow jobs: avoiding common anti-patterns, reading plans with `explain()`, broadcast joins, and caching. Partitioning, data skew and salting are summarised at the end as further reading.

All the data here is **generated inside Spark** (a million synthetic orders plus product and customer tables), so the notebook needs no files and behaves the same everywhere.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how the driver, executors, partitions and tasks work together.
- Read an execution plan with `explain()` and spot the shuffles (`Exchange`).
- Decide when to cache or persist a DataFrame, and release it afterwards.
- Use broadcast joins for large-to-small joins.
- Recognise and fix common performance anti-patterns (collecting large data, Python UDFs, recomputation).

## Prerequisites

- Notebook 01.02 (Spark and PySpark, and the walkthrough: transformations, aggregations, joins).
- Everything here runs on serverless compute; the one cell that changes a setting catches the error if your compute does not allow it.

## Index

1. [Why performance matters](#why)
2. [How Spark runs your code: a refresher](#how)
3. [Setup: generating test data](#setup)
4. [Common performance anti-patterns](#anti)
5. [Reading execution plans with explain()](#explain)
6. [Join optimisation: broadcast joins](#broadcast)
7. [Caching and persistence](#cache)
8. [Application: three reports from one pass](#application)
9. [Exercises](#exercises)
10. [Key Takeaways](#takeaways)
11. [Other topics for further reading](#other)
12. [Conclusion](#conclusion)
13. [Further Reading & Resources](#reading)

<a id="why"></a>
# Section 1: Why performance matters

Imagine a nightly pipeline that processes the day's transactions in **4 hours**. The business now wants the dashboards refreshed every hour. Without optimisation, that is simply impossible.

| Cost of a slow job | What it means in practice |
|---|---|
| Delayed insight | Decisions are made on stale data |
| Higher cloud bills | Clusters are paid for by the minute |
| Missed deadlines | Reports and regulatory submissions arrive late |
| Blocked pipelines | Downstream jobs wait for yours |
| Hidden scaling limits | It works on test data and fails on production volumes |

Almost every slow Spark job comes down to one of three causes (this notebook focuses on the first and third):

1. **Data movement (shuffles):** moving rows between machines over the network.
2. **Data skew:** a few partitions much larger than the rest, so one task runs long while others sit idle.
3. **Unnecessary work:** recomputing the same thing, pulling data to the driver, or using slow Python code where a built-in function exists.

<a id="how"></a>
# Section 2: How Spark runs your code: a refresher

**Definition.** A Spark application has one **driver** (the coordinator that plans the work and collects results) and several **executors** (worker processes that hold data and do the computation). Data is split into **partitions**; each partition is processed by one **task** on one executor core. A group of tasks that can run without exchanging data is a **stage**.

```
                    DRIVER  (plans the job, collects results)
                       |
        +--------------+--------------+
        v              v              v
   Executor 1     Executor 2     Executor 3
   [Part 1]       [Part 3]       [Part 5]
   [Part 2]       [Part 4]       [Part 6]
```

**Analogy.** A warehouse team packing orders. The supervisor (driver) splits the orders into crates (partitions) and hands them to packers (executors). Packing a crate needs nothing from other crates (a **narrow** step: filter, select, withColumn). But "put all orders for the same city together" means packers must walk crates across the floor to each other (a **shuffle**: groupBy, join, distinct, orderBy). The walking, not the packing, is what takes time.

**Explanation.** Two kinds of transformation:

| Narrow (cheap) | Wide (shuffle, expensive) |
|---|---|
| `select`, `filter`, `withColumn`, `union` | `groupBy`, `join` (non-broadcast), `distinct`, `orderBy`, `repartition` |
| Each output partition depends on one input partition | Rows with the same key must be brought together from every partition |
| Runs within a stage | Ends a stage: all tasks must finish before the next stage starts |

Shuffles are expensive because data is serialised, written to disk, sent over the network and read back. Network transfer is orders of magnitude slower than working in memory.

<a id="setup"></a>
# Section 3: Setup: generating test data

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import time
from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_02_04_PerformanceOptimisation").getOrCreate()
print("Spark version:", spark.version)

Two small helpers used throughout:

- `run(df)` forces Spark to execute a DataFrame fully **without** bringing results back, by writing to the special `noop` ("no operation") format. Useful for timing, because `count()` alone can let Spark skip work it does not need.
- `timed(label, df)` runs a DataFrame and prints how long it took.

**About timings:** on a small dataset or a busy shared cluster, timings are noisy and can even reverse. Treat them as illustrations; the **execution plan** is the reliable evidence of what Spark does.

In [ ]:
def run(df):
    """Execute the whole plan and discard the result."""
    try:
        # "noop" is a writer that computes every row and then throws it away: a pure timing tool
        df.write.format("noop").mode("overwrite").save()
    except Exception:
        df.agg(F.count(F.lit(1))).collect()   # fallback if the noop sink is unavailable

def timed(label, df):
    start = time.time()                       # wall-clock time before the job
    run(df)                                   # the action: Spark executes the plan now
    print(f"{label:<32} {time.time() - start:6.2f}s")

Now the data: **1,000,000 orders**, **5,000 products** and **100,000 customers**, generated with `spark.range` and seeded random numbers so the result is identical on every run. The customer ids are deliberately **skewed**: 80% of orders come from the first 1,000 customers, as in many real businesses (skew is covered in *Other topics for further reading*, after the Key Takeaways).

In [ ]:
N_ORDERS = 1_000_000

# pick() chooses one value from a list at random for every row: build an array column of the
# values, draw a random position 1..len(values), and take the element at that position.
def pick(values, seed):
    """A random choice from a list of strings, as a Spark column."""
    return F.element_at(F.array(*[F.lit(v) for v in values]), (F.rand(seed) * len(values)).cast("int") + 1)

# spark.range(n) creates a DataFrame with one column, id = 0..n-1, generated in parallel.
# Every other column is derived from it with seeded random numbers (rand(seed)), so each run is identical.
orders = (
    spark.range(N_ORDERS)
    .withColumn("order_id", F.format_string("ORD-%08d", F.col("id")))
    .withColumn("customer_id",
                F.when(F.rand(1) < 0.8, (F.rand(2) * 1000).cast("int") + 1)      # 80% of orders: customers 1-1000
                 .otherwise((F.rand(3) * 99000).cast("int") + 1001))              # 20%: customers 1001-100000
    .withColumn("product_id", (F.rand(4) * 5000).cast("int") + 1)
    .withColumn("quantity", (F.rand(5) * 10).cast("int") + 1)
    .withColumn("unit_price", F.round(F.rand(6) * 495 + 5, 2))
    .withColumn("order_date", F.expr("date_add(DATE'2024-01-01', CAST(rand(7) * 366 AS INT))"))
    .withColumn("region", pick(["North", "South", "East", "West"], 8))
    .withColumn("channel", pick(["Online", "Store", "Phone", "Mobile"], 9))
    .drop("id")
)
products = (
    spark.range(1, 5001).withColumnRenamed("id", "product_id")
    .withColumn("product_name", F.format_string("Product_%d", F.col("product_id")))
    .withColumn("category", pick(["Electronics", "Clothing", "Home", "Sports", "Books"], 10))
)
customers = (
    spark.range(1, 100001).withColumnRenamed("id", "customer_id")
    .withColumn("customer_name", F.format_string("Customer_%d", F.col("customer_id")))
    .withColumn("tier", pick(["Bronze", "Silver", "Gold", "Platinum"], 11))
)

print(f"Orders: {orders.count():,} | Products: {products.count():,} | Customers: {customers.count():,}")
orders.show(5)

<a id="anti"></a>
# Section 4: Common performance anti-patterns

An **anti-pattern** is a habit that seems reasonable but hurts performance. Three come up again and again, and they are the easiest wins of all.

**Anti-pattern 1: collecting large data.** `collect()` and `toPandas()` bring every row to the driver, a single machine. On big data that means an out-of-memory crash. Aggregate or sample in Spark first, and bring back only small results.

In [ ]:
# BAD:  all_rows = orders.collect()          # one million rows into the driver's memory
# GOOD: aggregate or limit in Spark, then collect the small result

**Anti-pattern 2: Python UDFs where a built-in function exists.** A **UDF** (user-defined function) runs your own Python code on each row. To do that Spark must ship every row from the JVM to a Python process and back, which is far slower than its built-in functions, and the optimiser cannot see inside a UDF. Use `pyspark.sql.functions` whenever possible; if you truly need custom Python, a vectorised **pandas UDF** is much faster than a row-by-row UDF.

**What "vectorised" means.** A plain UDF is called once **per row**, with one value. A **vectorised UDF** (a pandas UDF, written with `@F.pandas_udf`) is called once **per batch** of thousands of rows, with a whole pandas Series, and Spark moves each batch to Python in one go using **Apache Arrow** (a fast columnar format). Far fewer trips between the JVM and Python make it much faster than a plain UDF, but it is still slower than a built-in function, which never leaves the engine.

The cell below writes the same rule (`"Large"` if the quantity is 8 or more, otherwise `"Small"`) three ways and times each on the million orders.

In [ ]:
# 1. A plain Python UDF: Spark calls size_udf once for EVERY row. Each quantity is converted to a
#    Python object, sent from the JVM to a Python worker, and the answer is sent back: a million trips.
# 2. A vectorised (pandas) UDF: Spark sends a whole BATCH of quantities at once, as a pandas Series
#    (via Apache Arrow). The function handles the batch in one call and returns a Series of answers.
# 3. The built-in function: the rule is part of Spark's own plan, so no data ever goes to Python.
# Time the three versions (timed() runs the whole plan; see Section 3). Some locked-down computes
# block Python UDFs, so the first two are wrapped in try / except.

**Reading the timings.** The built-in version should be the fastest by a wide margin, on any compute. The order of the two UDFs depends on the machine: on a cluster the pandas UDF is usually several times faster than the plain UDF, but on a small or busy machine the cost of preparing each batch can hide the difference, or even reverse it (see the note on noisy timings in Section 3). The lesson does not change: built-in first, a pandas UDF if you really need Python, a plain UDF last.

**Anti-pattern 3: recomputing reused results.** Every action on an uncached DataFrame re-runs its whole plan. If an expensive intermediate result feeds several outputs, cache it once (Section 7), or restructure so that one pass produces everything (Section 8).

<a id="explain"></a>
# Section 5: Reading execution plans with explain()

**Definition.** `explain()` prints the **physical plan** Spark will use to compute a DataFrame, without running it: where data is read, which transformations are applied, and where data moves between executors.

**Example.** Total revenue by region.

**Analogy.** A flight plan filed before take-off: it tells you the route, the stops and the fuel, before a single passenger boards.

**Explanation.** Read plans **from the bottom up**: the source is at the bottom, the final result at the top. The operators to recognise:

| Operator | Meaning | Cost |
|---|---|---|
| `Scan` / `Range` | Read the source data | Depends on the source |
| `Project` | Keep or compute columns | Cheap (narrow) |
| `Filter` | Keep rows | Cheap (narrow), often pushed down to the source |
| `HashAggregate (partial_...)` | Pre-aggregate inside each partition | Cheap, reduces data before a shuffle |
| `Exchange hashpartitioning(...)` | **Shuffle** rows by key across executors | **Expensive** |
| `BroadcastExchange` | Copy a small table to every executor | Cheap for small tables |
| `SortMergeJoin` / `BroadcastHashJoin` | Join strategies | Sort-merge needs shuffles on both sides |
| `AdaptiveSparkPlan` | Adaptive Query Execution (AQE) may adjust the plan at run time | Usually helpful |

`explain()` shows the physical plan; `explain(mode="formatted")` splits it into a readable outline plus details. On Databricks and recent Spark versions **AQE** is on by default, so plans are wrapped in `AdaptiveSparkPlan isFinalPlan=false`: Spark may still change details (such as the number of shuffle partitions) once it sees the real data sizes.

In [ ]:
# Define the query (a transformation: nothing runs yet) ...
# ... and print the plan Spark would use to run it, still without running it

**Reading it bottom-up:**

1. **`Range`** (or `Scan`): generate / read the orders, one task per input partition.
2. **`Project`**: keep only `region`, `quantity`, `unit_price`; nothing else travels further.
3. **`HashAggregate ... partial_sum`**: each task sums revenue per region **for its own rows**. This map-side pre-aggregation means only a handful of partial sums leave each partition instead of all its rows.
4. **`Exchange hashpartitioning(region, ...)`**: the **shuffle**. Partial sums for the same region are sent to the same place. The number is the count of shuffle partitions (controlled by `spark.sql.shuffle.partitions`, 200 by default; AQE may reduce it).
5. **`HashAggregate ... sum`**: the final sums per region.

In [ ]:
# Your turn. Write your solution here:

Compare a query that needs a shuffle with one that does not:

In [ ]:
# groupBy is WIDE: rows with the same customer_id must meet, so look for "Exchange" in the plan
# filter is NARROW: each partition is filtered where it is, so there is no "Exchange"

#### Try it yourself

Before running anything, predict how many shuffles (`Exchange` operators) each query needs. Then check with `explain()`.

- Query A: orders in the North region.
- Query B: total quantity by region and channel.
- Query C: how many customers placed 1 order, 2 orders, 3 orders, ... (a count of counts).

In [ ]:
# Your turn

<a id="broadcast"></a>
# Section 6: Join optimisation: broadcast joins

**Definition.** In a **broadcast join**, Spark sends a full copy of the **small** table to every executor, so each partition of the large table can be joined locally. The large table is never shuffled.

**Example.** One million orders joined to five thousand products.

**Analogy.** Instead of making every shopper queue at a single information desk (shuffle), you give every shopper a printed copy of the store map (broadcast). Cheap, because the map is small.

**Explanation.** A normal join between two large tables is a **sort-merge join**: both sides are shuffled by the key so that matching rows meet. When one side is small, broadcasting it removes the shuffle of the big side entirely. Spark broadcasts automatically when a table's estimated size is below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default); `F.broadcast(df)` forces it when you know a table is small but Spark's estimate is not available or is wrong.

In [ ]:
# Let Spark choose the join strategy on its own
# F.broadcast(products) marks the SMALL table: a full copy goes to every executor, so the
# million orders are joined where they already are and never shuffled

Look for `BroadcastExchange` and `BroadcastHashJoin` in the second plan, and the absence of an `Exchange hashpartitioning` above the orders scan. (Depending on the version and the size estimates, Spark may already choose a broadcast in the first plan on its own; the hint guarantees it.)

In [ ]:
# Tables smaller than this size (in bytes) are broadcast automatically
# Run both versions in full and compare (timings are noisy; the plans above are the evidence)

Never broadcast a large table: every executor must hold a full copy in memory, and the driver must collect it first. A few hundred MB is a practical upper limit.

<a id="cache"></a>
# Section 7: Caching and persistence

**Definition.** **Caching** (`df.cache()`) asks Spark to keep a DataFrame's computed result in memory after the first action, so later actions reuse it instead of recomputing the whole plan from the source. `persist(level)` does the same with a chosen storage level. `unpersist()` releases it.

**Example.** A filtered DataFrame used by two different reports.

**Analogy.** Photocopying a page you will need several times, instead of fetching the original book from the archive every time.

**Explanation.** Because Spark is lazy, every action recomputes the plan from the start. Caching breaks that chain at a chosen point.

| Cache when | Do not cache when |
|---|---|
| The DataFrame is used **more than once** | It is used only once |
| It is **expensive** to compute (joins, wide aggregations) | It is too large for the cluster's memory |
| It fits in memory | The source is already fast (e.g. a small table) |

Caching is itself lazy: the data is stored by the **first action** after `cache()`. Always `unpersist()` when finished, so memory is free for other work.

> **Serverless note:** `cache()`, `persist()` and `unpersist()` work on serverless compute, but `checkpoint()`, `spark.catalog.cacheTable()` and `CACHE TABLE` do not. The cell below still catches any error, in case your compute restricts caching.

In [ ]:
# An intermediate result that two actions below both need
# Without a cache: each action re-runs the plan from the start (generate the orders, filter, compute)
# With a cache: the first action computes north once and keeps it in memory; the second reuses it

<a id="application"></a>
# Section 8: Application: three reports from one pass

**Scenario.** Every morning, three reports are produced from the same enriched orders: revenue by region, revenue by category, and the top five products. The current code builds each report from scratch, so the join between orders and products runs **three times**.

**Process.** Identify the shared work, compute it once with a broadcast join, reuse it, and compare.

In [ ]:
# The current code: three independent functions, each repeating the same join and the same
# revenue calculation, so the shared work is done three times
def report_region():
    return orders.join(products, "product_id").groupBy("region").agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue"))

def report_category():
    return orders.join(products, "product_id").groupBy("category").agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue"))

def report_top5():
    return (orders.join(products, "product_id")
            .groupBy("product_id", "product_name").agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("revenue"))
            .orderBy(F.desc("revenue")).limit(5))

In [ ]:
# Your turn. Write your solution here:

In [ ]:
# 1. The shared work, once: broadcast the small table and compute the line total a single time
# 2. Reuse it (cache where the compute allows it)
# 3. The three reports now only aggregate the shared, already-computed result

<a id="exercises"></a>
# Exercises

### Exercise 1: Find and fix the anti-patterns

The function below works but is slow. Identify three anti-patterns, then write `good_code()` that returns the same results. (The two results are small, so collecting them is fine.)

```python
@F.udf(returnType=T.StringType())
def size_udf2(qty):
    return "Large" if qty >= 8 else "Small"

def bad_code():
    data = orders.join(products, "product_id")
    data = data.withColumn("size", size_udf2("quantity"))
    r1 = data.groupBy("category", "size").count().collect()
    r2 = data.groupBy("region", "size").count().collect()
    return r1, r2
```

In [ ]:
# Your turn

### Exercise 2: Plan detective

Without running it, how many shuffles does this query need, and why? Check with `explain()`, then rewrite it so that the join does not shuffle `orders`.

```python
q = (orders.join(customers, "customer_id")
           .groupBy("tier").agg(F.sum("quantity").alias("units")))
```

In [ ]:
# Your turn

<a id="takeaways"></a>
## Key Takeaways

**Performance checklist**, before running a heavy job:

1. **Keep data in Spark:** never `collect()` or `toPandas()` large DataFrames.
2. **Prefer built-in functions** to Python UDFs.
3. **Check the plan:** `explain()` and count the `Exchange` operators (shuffles).
4. **Broadcast small tables** in joins with large ones.
5. **Reuse expensive results:** cache what is used more than once, and `unpersist()` afterwards.
6. **Still slow?** Read *Other topics for further reading* below: partitioning, skew and salting.

| Command | Purpose |
|---|---|
| `df.explain()` / `df.explain(mode="formatted")` | Show the physical plan |
| `F.broadcast(small_df)` | Force a broadcast join |
| `df.cache()` / `df.persist(level)` / `df.unpersist()` | Keep or release a computed result |
| `df.write.format("noop").mode("overwrite").save()` | Run a plan fully without output (timing) |

<a id="other"></a>
## Other topics for further reading

The topics below matter on very large or unevenly distributed data. On Databricks, **Adaptive Query Execution** (AQE) handles much of this automatically, so they are not part of the lesson. Read them when a job is still slow after the checklist in the Key Takeaways.

**Partitioning: `repartition()` versus `coalesce()`.** Spark splits a DataFrame into **partitions**, one task each. Too few partitions leave cores idle; thousands of tiny ones waste time on scheduling. `repartition(n)` (or `repartition("col")`) redistributes the rows with a full shuffle, giving even sizes; `coalesce(n)` only merges existing partitions, without a shuffle, and is the usual way to write **fewer output files** (for example `df.coalesce(1).write...` for one small CSV). Rule of thumb: about 2 to 4 partitions per CPU core, or about 128 MB per partition. *Read:* [Spark SQL performance tuning: coalesce and repartition hints](https://spark.apache.org/docs/latest/sql-performance-tuning.html).

**Data skew.** When one key holds most of the rows (one huge customer, one default value such as a "guest" id), the task handling that key runs long after the others have finished: the job is only as fast as its slowest task. Measure it by counting rows per partition with `F.spark_partition_id()`, or rows per key with a `groupBy`. AQE splits skewed join partitions automatically (`spark.sql.adaptive.skewJoin.enabled`). *Read:* [Adaptive Query Execution](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution).

**Salting.** The manual fix for a skewed key: add a random suffix (the *salt*, 0 to N-1) to the key in the big table, repeat each row of the small table N times (once per suffix), and join on the salted key, so the hot key is spread over N tasks. For a skewed **aggregation**, aggregate by key and salt first, then add the partial results up by key. The cost is a small table N times bigger, so keep N small. *Read:* [Databricks: skew join optimisation](https://docs.databricks.com/aws/en/optimizations/skew-join).

**Persistence levels.** `cache()` keeps data in memory and spills to disk if needed (`MEMORY_AND_DISK`). `persist(StorageLevel.MEMORY_ONLY)` or `DISK_ONLY` choose explicitly. The default is right in almost every case. *Read:* [RDD persistence and storage levels](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-persistence).

**Tuning shuffle partitions.** `spark.sql.shuffle.partitions` (default 200) sets how many pieces a `groupBy` or `join` produces. 200 is too many for small data and too few for very large data; AQE merges small shuffle partitions automatically, which is why you rarely need to change it on Databricks. It can be set per session with `spark.conf.set("spark.sql.shuffle.partitions", "64")` where your compute allows it. *Read:* [Spark configuration: shuffle behaviour](https://spark.apache.org/docs/latest/configuration.html#shuffle-behavior).

<a id="conclusion"></a>
## Conclusion

You can now look inside Spark's execution plans, recognise the expensive steps (the shuffles), and apply the standard fixes: avoiding the classic anti-patterns, broadcast joins and caching, with partitioning, skew and salting ready for when you need them. These habits matter even more in the MLlib notebooks, where machine-learning pipelines read the same data many times.

<a id="reading"></a>
## Further Reading & Resources

- Apache Spark: [Spark SQL performance tuning (caching, AQE, join hints, skew)](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
- Apache Spark: [Tuning guide (serialisation, memory)](https://spark.apache.org/docs/latest/tuning.html)
- Apache Spark: [DataFrame.explain](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
- Databricks: [Performance efficiency best practices](https://docs.databricks.com/aws/en/lakehouse-architecture/performance-efficiency/best-practices)
- Databricks: [Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations)